In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP16
# TCGA-LIHC EXTERNAL VALIDATION MODEL FREEZE
#
# Purpose:
# 1. Freeze model structure after internal holdout validation
# 2. Refit fixed models on full TCGA N=340
# 3. Create portable model specifications
# 4. Prepare for GEO / ICGC external validation
#
# IMPORTANT:
# NO feature reselection
# NO algorithm reselection
# NO external tuning
# ============================================================

from pathlib import Path
from datetime import datetime

import json
import hashlib
import shutil
import warnings
import sys

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


ROOT = Path(_hcc_path(''))

STEP15_DIR = (
    ROOT /
    "LIHC_RNA_Step15_Machine_Learning"
)

HOLDOUT_DIR = (
    ROOT /
    "LIHC_RNA_Step15_Machine_Learning_Holdout_Validation"
)

PROJECT_DIR = (
    ROOT /
    "LIHC_RNA_Step16_External_Validation_Freeze"
)

INPUT_DIR = (
    PROJECT_DIR /
    "Input"
)

RESULTS_DIR = (
    PROJECT_DIR /
    "Results"
)

FIGURES_DIR = (
    PROJECT_DIR /
    "Figures"
)

MODELS_DIR = (
    PROJECT_DIR /
    "Models"
)

AUDIT_DIR = (
    PROJECT_DIR /
    "Audit"
)

NOTEBOOK_DIR = (
    PROJECT_DIR /
    "Notebook"
)

for d in [
    INPUT_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    MODELS_DIR,
    AUDIT_DIR,
    NOTEBOOK_DIR
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )


RUN_TIME = (
    datetime.now()
    .strftime("%Y-%m-%d %H:%M:%S")
)

VERSION = (
    "ExternalValidationModel_v2.0"
)

RANDOM_STATE = 20260921


print("=" * 80)
print("STEP16 EXTERNAL VALIDATION MODEL FREEZE")
print("=" * 80)
print("Version:", VERSION)
print("Run:", RUN_TIME)
print("Project:", PROJECT_DIR)

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from lifelines import CoxPHFitter

from sksurv.util import Surv

from sksurv.linear_model import (
    CoxPHSurvivalAnalysis,
    CoxnetSurvivalAnalysis
)

import sklearn
import sksurv
import lifelines
import scipy
import joblib


print("Python:", sys.version)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)
print("sksurv:", sksurv.__version__)
print("lifelines:", lifelines.__version__)

In [ ]:
# ============================================================
# COPY LOCKED INPUT FILES
# ============================================================

source_files = {

    "LIHC_Multilevel_Master_Feature_Table.csv":
        STEP15_DIR /
        "Input" /
        "LIHC_Multilevel_Master_Feature_Table.csv",

    "LIHC_Survival_Train_Test_Split_70_30.csv":
        STEP15_DIR /
        "Input" /
        "LIHC_Survival_Train_Test_Split_70_30.csv",

    "PHASE_B_HOLDOUT_VALIDATION_AUDIT.md":
        HOLDOUT_DIR /
        "Audit" /
        "PHASE_B_HOLDOUT_VALIDATION_AUDIT.md",

    "04_Holdout_Paired_Delta_CIndex.csv":
        HOLDOUT_DIR /
        "Results" /
        "Holdout_Validation" /
        "04_Holdout_Paired_Delta_CIndex.csv",

    "18_FINAL_Holdout_Model_Performance.csv":
        HOLDOUT_DIR /
        "Results" /
        "Holdout_Validation" /
        "18_FINAL_Holdout_Model_Performance.csv",
}


manifest_records = []


for filename, src in source_files.items():

    if not src.exists():

        raise FileNotFoundError(
            f"Missing locked input:\n{src}"
        )

    dst = (
        INPUT_DIR /
        filename
    )

    shutil.copy2(
        src,
        dst
    )

    manifest_records.append({
        "File":
            filename,

        "Original_Path":
            str(src),

        "Step16_Copy":
            str(dst),

        "Size_KB":
            round(
                dst.stat().st_size
                / 1024,
                2
            )
    })


input_manifest = pd.DataFrame(
    manifest_records
)

input_manifest.to_csv(
    AUDIT_DIR /
    "01_Input_File_Manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    input_manifest
)

In [ ]:
# ============================================================
# INPUT SHA256
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
                f.read(1024 * 1024),
            b""
        ):

            h.update(
                chunk
            )

    return h.hexdigest()


hash_records = []


for path in sorted(
    INPUT_DIR.iterdir()
):

    if path.is_file():

        hash_records.append({
            "File":
                path.name,

            "SHA256":
                sha256_file(
                    path
                )
        })


hash_df = pd.DataFrame(
    hash_records
)

hash_df.to_csv(
    AUDIT_DIR /
    "02_Input_SHA256.csv",
    index=False,
    encoding="utf-8-sig"
)

display(hash_df)

In [ ]:
# ============================================================
# LOCK MODEL STRUCTURE
# ============================================================

ORIGINAL_TOP10 = [
    "HOXD9",
    "RFESD",
    "RPSAP53",
    "TRGC1",
    "FAM227B",
    "AC015871.6",
    "LINC00324",
    "ACOXL",
    "PTMAP4",
    "ANKRD20A7P"
]


DYNAMIC9 = [
    "HOXD9",
    "RFESD",
    "RPSAP53",
    "TRGC1",
    "FAM227B",
    "LINC00324",
    "PTMAP4",
    "ANKRD20A7P",
    "BOP1"
]


MODEL_STRUCTURE = pd.DataFrame([
    {
        "Model_ID":
            "EXT01_OriginalTop10_Cox",

        "Model_Role":
            "PRIMARY",

        "Feature_Set":
            "Original Top10",

        "Algorithm":
            "Unpenalized CoxPH",

        "N_Features":
            10,

        "Feature_Selection_Status":
            "LOCKED",

        "External_Retuning":
            "PROHIBITED"
    },

    {
        "Model_ID":
            "EXT02_OriginalTop10_ElasticNet",

        "Model_Role":
            "SENSITIVITY",

        "Feature_Set":
            "Original Top10",

        "Algorithm":
            "ElasticNet Cox",

        "N_Features":
            10,

        "Feature_Selection_Status":
            "LOCKED",

        "External_Retuning":
            "PROHIBITED"
    },

    {
        "Model_ID":
            "EXT03_Dynamic9_Cox",

        "Model_Role":
            "ALTERNATIVE",

        "Feature_Set":
            "Dynamic9",

        "Algorithm":
            "Unpenalized CoxPH",

        "N_Features":
            9,

        "Feature_Selection_Status":
            "LOCKED",

        "External_Retuning":
            "PROHIBITED"
    }
])


MODEL_STRUCTURE.to_csv(
    RESULTS_DIR /
    "01_STEP16_Locked_Model_Structure.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    MODEL_STRUCTURE
)

In [ ]:
# ============================================================
# DYNAMIC9 LINEAGE
# ============================================================

dynamic9_lineage = pd.DataFrame([
    {
        "Feature":
            gene,

        "Source":
            (
                "Strict 3-of-3 intersection from "
                "Development N=238"
            ),

        "Candidate_Pool":
            (
                "Original Top10 + strict WGCNA Hub18"
            ),

        "Candidate_Pool_N":
            28,

        "Method_1":
            "Univariate Cox P<0.05",

        "Method_2":
            (
                "ElasticNet Cox with "
                "Development-only inner CV"
            ),

        "Method_3":
            (
                "Random Survival Forest "
                "permutation importance Top10"
            ),

        "Holdout_Used_For_Selection":
            "NO"
    }

    for gene in DYNAMIC9
])


dynamic9_lineage.to_csv(
    RESULTS_DIR /
    "02_Dynamic9_Feature_Lineage.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    dynamic9_lineage
)

In [ ]:
# ============================================================
# BUILD FULL TCGA COHORT
# ============================================================

master = pd.read_csv(
    INPUT_DIR /
    "LIHC_Multilevel_Master_Feature_Table.csv"
)

split_df = pd.read_csv(
    INPUT_DIR /
    "LIHC_Survival_Train_Test_Split_70_30.csv"
)


def find_id_column(df):

    candidates = [
        "Sample_ID",
        "sample_id",
        "Sample",
        "ID"
    ]

    for c in candidates:

        if c in df.columns:
            return c

    raise KeyError(
        "Sample ID column not found."
    )


MASTER_ID = find_id_column(
    master
)

SPLIT_ID = find_id_column(
    split_df
)


master[
    MASTER_ID
] = (
    master[
        MASTER_ID
    ]
    .astype(str)
    .str.strip()
)

split_df[
    SPLIT_ID
] = (
    split_df[
        SPLIT_ID
    ]
    .astype(str)
    .str.strip()
)


full_tcga = (
    split_df[
        [
            SPLIT_ID,
            "Dataset"
        ]
    ]
    .merge(
        master,
        left_on=SPLIT_ID,
        right_on=MASTER_ID,
        how="left",
        validate="one_to_one"
    )
)


if len(full_tcga) != 340:

    raise RuntimeError(
        f"Expected N=340, got {len(full_tcga)}"
    )


print(
    "Full TCGA N:",
    len(full_tcga)
)

print(
    full_tcga[
        "Dataset"
    ].value_counts()
)

In [ ]:
# ============================================================
# OUTCOME AUDIT
# ============================================================

TIME_COL = "OS_time"
EVENT_COL = "OS"


full_tcga[
    TIME_COL
] = pd.to_numeric(
    full_tcga[
        TIME_COL
    ],
    errors="coerce"
)

full_tcga[
    EVENT_COL
] = pd.to_numeric(
    full_tcga[
        EVENT_COL
    ],
    errors="coerce"
)


qc = pd.DataFrame([
    {
        "N":
            len(full_tcga),

        "Events":
            int(
                full_tcga[
                    EVENT_COL
                ].sum()
            ),

        "Missing_Time":
            int(
                full_tcga[
                    TIME_COL
                ].isna()
                .sum()
            ),

        "Missing_Event":
            int(
                full_tcga[
                    EVENT_COL
                ].isna()
                .sum()
            )
    }
])


qc.to_csv(
    RESULTS_DIR /
    "03_TCGA340_Outcome_QC.csv",
    index=False,
    encoding="utf-8-sig"
)

display(qc)

In [ ]:
# ============================================================
# SURVIVAL FUNCTION
# ============================================================

def make_y(df):

    return Surv.from_arrays(

        event=(
            df[
                EVENT_COL
            ]
            .astype(int)
            .astype(bool)
            .values
        ),

        time=(
            df[
                TIME_COL
            ]
            .astype(float)
            .values
        )
    )

In [ ]:
# ============================================================
# FIT FINAL PRIMARY MODEL ON TCGA N=340
# ============================================================

PRIMARY_FEATURES = (
    ORIGINAL_TOP10
)


X_raw = (
    full_tcga[
        PRIMARY_FEATURES
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)


primary_imputer = SimpleImputer(
    strategy="median"
)

X_imp = (
    primary_imputer
    .fit_transform(
        X_raw
    )
)


primary_scaler = StandardScaler()

X_std = (
    primary_scaler
    .fit_transform(
        X_imp
    )
)


primary_model = CoxPHSurvivalAnalysis(
    alpha=0.0
)

primary_model.fit(
    X_std,
    make_y(
        full_tcga
    )
)


primary_risk = (
    primary_model
    .predict(
        X_std
    )
)


print(
    "Primary model fitted on N=",
    len(full_tcga)
)

In [ ]:
# ============================================================
# LOCK EXTERNAL VALIDATION CUTOFF
# ============================================================

EXTERNAL_PRIMARY_CUTOFF = float(
    np.median(
        primary_risk
    )
)


cutoff_df = pd.DataFrame([
    {
        "Model_ID":
            "EXT01_OriginalTop10_Cox",

        "Training_Cohort":
            "TCGA-LIHC N=340",

        "Cutoff_Method":
            "Median linear predictor",

        "Cutoff_Value":
            EXTERNAL_PRIMARY_CUTOFF,

        "External_Reoptimization":
            "NO"
    }
])


cutoff_df.to_csv(
    RESULTS_DIR /
    "04_External_Model_Cutoff_LOCKED.csv",
    index=False,
    encoding="utf-8-sig"
)


print(
    "FINAL external-validation cutoff:",
    EXTERNAL_PRIMARY_CUTOFF
)

In [ ]:
# ============================================================
# PORTABLE PRIMARY PARAMETERS
# ============================================================

primary_coef = np.asarray(
    primary_model.coef_
).reshape(-1)


primary_parameters = pd.DataFrame({
    "Feature":
        PRIMARY_FEATURES,

    "Coefficient":
        primary_coef,

    "Imputation_Median":
        primary_imputer.statistics_,

    "Scaling_Mean":
        primary_scaler.mean_,

    "Scaling_SD":
        primary_scaler.scale_
})


primary_parameters.to_csv(
    MODELS_DIR /
    "EXT01_OriginalTop10_Cox_Portable_Parameters.csv",
    index=False,
    encoding="utf-8-sig"
)


display(
    primary_parameters
)

In [ ]:
# ============================================================
# PRIMARY COEFFICIENT INFERENCE
# TCGA N=340
# ============================================================

lifelines_df = pd.DataFrame(
    X_std,
    columns=PRIMARY_FEATURES
)

lifelines_df[
    "OS_time"
] = (
    full_tcga[
        TIME_COL
    ].values
)

lifelines_df[
    "OS"
] = (
    full_tcga[
        EVENT_COL
    ].values
)


cph = CoxPHFitter(
    penalizer=0.0
)

cph.fit(
    lifelines_df,
    duration_col="OS_time",
    event_col="OS"
)


coef_table = (
    cph.summary
    .reset_index()
)


coef_table = coef_table.rename(
    columns={
        "covariate":
            "Gene",

        "coef":
            "Coefficient",

        "exp(coef)":
            "HR_per_1SD",

        "exp(coef) lower 95%":
            "HR_CI_Lower",

        "exp(coef) upper 95%":
            "HR_CI_Upper",

        "p":
            "P_Value"
    }
)


coef_table = coef_table[
    [
        "Gene",
        "Coefficient",
        "HR_per_1SD",
        "HR_CI_Lower",
        "HR_CI_Upper",
        "P_Value"
    ]
]


coef_table.to_csv(
    RESULTS_DIR /
    "05_EXT01_Final_Cox_Coefficients.csv",
    index=False,
    encoding="utf-8-sig"
)


display(
    coef_table
)

In [ ]:
# ============================================================
# COEFFICIENT / HR FOREST PLOT
# ============================================================

plot_df = (
    coef_table
    .sort_values(
        "HR_per_1SD"
    )
    .copy()
)


y = np.arange(
    len(plot_df)
)


x = (
    plot_df[
        "HR_per_1SD"
    ].values
)


xerr_lower = (
    x
    - plot_df[
        "HR_CI_Lower"
    ].values
)


xerr_upper = (
    plot_df[
        "HR_CI_Upper"
    ].values
    - x
)


fig, ax = plt.subplots(
    figsize=(8, 6)
)


ax.errorbar(
    x,
    y,
    xerr=[
        xerr_lower,
        xerr_upper
    ],
    fmt="o",
    capsize=4
)


ax.axvline(
    1,
    linestyle="--"
)


ax.set_yticks(
    y
)


ax.set_yticklabels(
    plot_df[
        "Gene"
    ]
)


ax.set_xlabel(
    "Hazard ratio per 1-SD increase"
)


ax.set_title(
    "Final TCGA340 Original Top10 Cox Model"
)


fig.tight_layout()


fig.savefig(
    FIGURES_DIR /
    "01_EXT01_Final_Cox_HR_Forest.png",
    dpi=300,
    bbox_inches="tight"
)


fig.savefig(
    FIGURES_DIR /
    "01_EXT01_Final_Cox_HR_Forest.svg",
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# FINAL TCGA340 RISK SCORES
# ============================================================

risk_df = pd.DataFrame({

    "Sample_ID":
        full_tcga[
            MASTER_ID
        ].astype(str).values,

    "Historical_Dataset":
        full_tcga[
            "Dataset"
        ].values,

    "OS_time":
        full_tcga[
            TIME_COL
        ].values,

    "OS":
        full_tcga[
            EVENT_COL
        ].values,

    "ExternalValidation_RiskScore":
        primary_risk,

    "ExternalValidation_RiskGroup":
        np.where(
            primary_risk
            >= EXTERNAL_PRIMARY_CUTOFF,
            "High",
            "Low"
        )
})


risk_df.to_csv(
    RESULTS_DIR /
    "06_TCGA340_Final_Primary_RiskScores.csv",
    index=False,
    encoding="utf-8-sig"
)


display(
    risk_df.head()
)

In [ ]:
# ============================================================
# RISK DISTRIBUTION FIGURE
# ============================================================

risk_plot = (
    risk_df
    .sort_values(
        "ExternalValidation_RiskScore"
    )
    .reset_index(drop=True)
)


risk_plot[
    "Rank"
] = np.arange(
    1,
    len(risk_plot) + 1
)


fig, ax = plt.subplots(
    figsize=(10, 5)
)


ax.scatter(
    risk_plot[
        "Rank"
    ],
    risk_plot[
        "ExternalValidation_RiskScore"
    ],
    s=22
)


ax.axhline(
    EXTERNAL_PRIMARY_CUTOFF,
    linestyle="--",
    label="Locked TCGA340 median cutoff"
)


ax.set_xlabel(
    "Patients ranked by risk score"
)


ax.set_ylabel(
    "Final linear predictor"
)


ax.set_title(
    "TCGA340 Final External-Validation Model Risk Distribution"
)


ax.legend()


fig.tight_layout()


fig.savefig(
    FIGURES_DIR /
    "02_TCGA340_Final_Risk_Distribution.png",
    dpi=300,
    bbox_inches="tight"
)


fig.savefig(
    FIGURES_DIR /
    "02_TCGA340_Final_Risk_Distribution.svg",
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# BASELINE SURVIVAL
# ============================================================

baseline_fn = (
    primary_model
    .baseline_survival_
)


baseline_df = pd.DataFrame({
    "Time_Days":
        baseline_fn.x,

    "Baseline_Survival":
        baseline_fn.y
})


baseline_df.to_csv(
    MODELS_DIR /
    "EXT01_Baseline_Survival_Function.csv",
    index=False,
    encoding="utf-8-sig"
)


timepoints = [
    365,
    730,
    1095
]


baseline_points = []


for t in timepoints:

    baseline_points.append({
        "Time_Days":
            t,

        "Baseline_Survival":
            float(
                baseline_fn(t)
            )
    })


baseline_points = pd.DataFrame(
    baseline_points
)


baseline_points.to_csv(
    MODELS_DIR /
    "EXT01_Baseline_Survival_1_2_3Y.csv",
    index=False,
    encoding="utf-8-sig"
)


display(
    baseline_points
)

In [ ]:
# ============================================================
# PRIMARY MODEL BUNDLE
# ============================================================

PRIMARY_BUNDLE = {

    "Model_ID":
        "EXT01_OriginalTop10_Cox",

    "Version":
        VERSION,

    "Model_Role":
        "PRIMARY",

    "Training_Cohort":
        "TCGA-LIHC N=340",

    "Features":
        PRIMARY_FEATURES,

    "Algorithm":
        "Unpenalized Cox proportional hazards",

    "Model":
        primary_model,

    "Imputer":
        primary_imputer,

    "Scaler":
        primary_scaler,

    "Risk_Cutoff":
        EXTERNAL_PRIMARY_CUTOFF,

    "Cutoff_Rule":
        "Median TCGA340 linear predictor",

    "External_Cutoff_Reoptimization":
        False,

    "External_Feature_Selection":
        False,

    "External_Hyperparameter_Tuning":
        False,

    "Time_Unit":
        "days"
}


PRIMARY_MODEL_FILE = (
    MODELS_DIR /
    "EXT01_OriginalTop10_Cox_ExternalValidationModel_v2.joblib"
)


joblib.dump(
    PRIMARY_BUNDLE,
    PRIMARY_MODEL_FILE
)


print(
    PRIMARY_MODEL_FILE
)

In [ ]:
# ============================================================
# PORTABLE JSON MODEL SPECIFICATION
# ============================================================

portable_spec = {

    "Model_ID":
        "EXT01_OriginalTop10_Cox",

    "Version":
        VERSION,

    "Role":
        "PRIMARY",

    "Training_Cohort":
        "TCGA-LIHC N=340",

    "Features":
        PRIMARY_FEATURES,

    "Algorithm":
        "Unpenalized Cox proportional hazards",

    "Preprocessing":
        (
            "Median imputation using TCGA340 parameters; "
            "z-score using TCGA340 mean and SD"
        ),

    "Risk_Formula":
        (
            "LP = sum(beta_i * "
            "((x_i - mean_i) / SD_i))"
        ),

    "Risk_Cutoff":
        EXTERNAL_PRIMARY_CUTOFF,

    "Cutoff_Rule":
        "TCGA340 median linear predictor",

    "External_Reoptimization":
        "PROHIBITED",

    "External_Feature_Reselection":
        "PROHIBITED",

    "External_Coefficient_Refitting":
        "PROHIBITED",

    "Time_Unit":
        "days"
}


with open(
    MODELS_DIR /
    "EXT01_OriginalTop10_Cox_Portable_Specification.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        portable_spec,
        f,
        ensure_ascii=False,
        indent=2
    )

In [ ]:
# ============================================================
# FREEZE DYNAMIC9 ALTERNATIVE MODEL
# ============================================================

X_dyn_raw = (
    full_tcga[
        DYNAMIC9
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)


dyn_imputer = SimpleImputer(
    strategy="median"
)

X_dyn_imp = (
    dyn_imputer
    .fit_transform(
        X_dyn_raw
    )
)


dyn_scaler = StandardScaler()

X_dyn_std = (
    dyn_scaler
    .fit_transform(
        X_dyn_imp
    )
)


dyn_model = CoxPHSurvivalAnalysis(
    alpha=0.0
)


dyn_model.fit(
    X_dyn_std,
    make_y(
        full_tcga
    )
)


dyn_risk = (
    dyn_model
    .predict(
        X_dyn_std
    )
)


DYNAMIC9_CUTOFF = float(
    np.median(
        dyn_risk
    )
)


dyn_parameters = pd.DataFrame({

    "Feature":
        DYNAMIC9,

    "Coefficient":
        np.asarray(
            dyn_model.coef_
        ).reshape(-1),

    "Imputation_Median":
        dyn_imputer.statistics_,

    "Scaling_Mean":
        dyn_scaler.mean_,

    "Scaling_SD":
        dyn_scaler.scale_
})


dyn_parameters.to_csv(
    MODELS_DIR /
    "EXT03_Dynamic9_Cox_Portable_Parameters.csv",
    index=False,
    encoding="utf-8-sig"
)


DYNAMIC9_BUNDLE = {

    "Model_ID":
        "EXT03_Dynamic9_Cox",

    "Version":
        VERSION,

    "Model_Role":
        "ALTERNATIVE",

    "Features":
        DYNAMIC9,

    "Model":
        dyn_model,

    "Imputer":
        dyn_imputer,

    "Scaler":
        dyn_scaler,

    "Risk_Cutoff":
        DYNAMIC9_CUTOFF,

    "External_Retuning":
        False
}


joblib.dump(
    DYNAMIC9_BUNDLE,
    MODELS_DIR /
    "EXT03_Dynamic9_Cox_ExternalValidationModel_v2.joblib"
)


print(
    "Dynamic9 frozen."
)

print(
    "Cutoff:",
    DYNAMIC9_CUTOFF
)

In [ ]:
# ============================================================
# FINAL EXTERNAL MODEL REGISTRY
# ============================================================

external_registry = pd.DataFrame([
    {
        "Model_ID":
            "EXT01_OriginalTop10_Cox",

        "Role":
            "PRIMARY",

        "N_Features":
            10,

        "Algorithm":
            "Unpenalized CoxPH",

        "Training_N":
            340,

        "Risk_Cutoff":
            EXTERNAL_PRIMARY_CUTOFF,

        "Ready_For_External_Validation":
            "YES"
    },

    {
        "Model_ID":
            "EXT03_Dynamic9_Cox",

        "Role":
            "ALTERNATIVE",

        "N_Features":
            9,

        "Algorithm":
            "Unpenalized CoxPH",

        "Training_N":
            340,

        "Risk_Cutoff":
            DYNAMIC9_CUTOFF,

        "Ready_For_External_Validation":
            "YES"
    }
])


external_registry.to_csv(
    RESULTS_DIR /
    "07_External_Validation_Model_Registry.csv",
    index=False,
    encoding="utf-8-sig"
)


display(
    external_registry
)

In [ ]:
# ============================================================
# HUMAN-READABLE MODEL FORMULA
# ============================================================

formula_lines = []

formula_lines.append(
    "EXT01 Original Top10 Cox Model"
)

formula_lines.append(
    "=" * 80
)

formula_lines.append(
    "Standardization:"
)

formula_lines.append(
    "z_i = (x_i - TCGA340_mean_i) / TCGA340_SD_i"
)

formula_lines.append(
    ""
)

formula_lines.append(
    "Linear predictor:"
)


for feature, beta in zip(
    PRIMARY_FEATURES,
    primary_coef
):

    formula_lines.append(
        f"{feature}: beta = {beta:.10f}"
    )


formula_lines.append(
    ""
)

formula_lines.append(
    "RiskScore = sum(beta_i * z_i)"
)

formula_lines.append(
    ""
)

formula_lines.append(
    f"High-risk cutoff = "
    f"{EXTERNAL_PRIMARY_CUTOFF:.10f}"
)

formula_lines.append(
    ""
)

formula_lines.append(
    "External re-optimization: PROHIBITED"
)


formula_text = "\n".join(
    formula_lines
)


with open(
    MODELS_DIR /
    "EXT01_Human_Readable_Model_Formula.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        formula_text
    )


print(
    formula_text
)

In [ ]:
# ============================================================
# EXTERNAL VALIDATION PROTOCOL
# ============================================================

protocol = [
    {
        "Step": 1,
        "Rule":
            "Confirm independent external cohort"
    },

    {
        "Step": 2,
        "Rule":
            "Map the fixed Original Top10 genes"
    },

    {
        "Step": 3,
        "Rule":
            (
                "Do not perform outcome-guided "
                "feature filtering"
            )
    },

    {
        "Step": 4,
        "Rule":
            (
                "Apply frozen TCGA340 preprocessing "
                "parameters"
            )
    },

    {
        "Step": 5,
        "Rule":
            (
                "Apply frozen TCGA340 Cox "
                "coefficients"
            )
    },

    {
        "Step": 6,
        "Rule":
            (
                "Calculate linear predictor "
                "without coefficient refitting"
            )
    },

    {
        "Step": 7,
        "Rule":
            (
                "Apply frozen TCGA340 median cutoff"
            )
    },

    {
        "Step": 8,
        "Rule":
            (
                "Evaluate C-index with "
                "bootstrap 95% CI"
            )
    },

    {
        "Step": 9,
        "Rule":
            (
                "Evaluate time-dependent AUC "
                "when endpoint permits"
            )
    },

    {
        "Step": 10,
        "Rule":
            (
                "Evaluate calibration without "
                "recalibrating first"
            )
    },

    {
        "Step": 11,
        "Rule":
            "Evaluate KM / HR using frozen cutoff"
    },

    {
        "Step": 12,
        "Rule":
            (
                "Any recalibration must be clearly "
                "reported as secondary analysis"
            )
    }
]


protocol_df = pd.DataFrame(
    protocol
)


protocol_df.to_csv(
    RESULTS_DIR /
    "08_EXTERNAL_VALIDATION_PROTOCOL_LOCKED.csv",
    index=False,
    encoding="utf-8-sig"
)


display(
    protocol_df
)

In [ ]:
# ============================================================
# SOFTWARE ENVIRONMENT
# ============================================================

environment_info = {

    "Python":
        sys.version,

    "numpy":
        np.__version__,

    "pandas":
        pd.__version__,

    "scikit-learn":
        sklearn.__version__,

    "scikit-survival":
        sksurv.__version__,

    "lifelines":
        lifelines.__version__,

    "scipy":
        scipy.__version__
}


with open(
    AUDIT_DIR /
    "03_SOFTWARE_ENVIRONMENT.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        environment_info,
        f,
        indent=2
    )

In [ ]:
# ============================================================
# STEP16 FINAL AUDIT
# ============================================================

audit = []

audit.append(
    "TCGA-LIHC STEP16 EXTERNAL VALIDATION MODEL FREEZE"
)

audit.append(
    "=" * 80
)

audit.append(
    f"Version: {VERSION}"
)

audit.append(
    f"Run time: {RUN_TIME}"
)

audit.append(
    ""
)

audit.append(
    "MODEL SELECTION STATUS:"
)

audit.append(
    "Primary model selected BEFORE Step16."
)

audit.append(
    "No model reselection performed in Step16."
)

audit.append(
    ""
)

audit.append(
    "PRIMARY MODEL:"
)

audit.append(
    "EXT01 Original Top10 + Unpenalized CoxPH"
)

audit.append(
    ""
)

audit.append(
    "Primary feature count: 10"
)

audit.append(
    "Training cohort: TCGA-LIHC N=340"
)

audit.append(
    ""
)

audit.append(
    "ALTERNATIVE MODEL:"
)

audit.append(
    "EXT03 Dynamic9 + Unpenalized CoxPH"
)

audit.append(
    ""
)

audit.append(
    "Dynamic9 feature selection source:"
)

audit.append(
    "Development N=238 only"
)

audit.append(
    "Candidate pool = OriginalTop10 + strict Hub18 (N=28)"
)

audit.append(
    "Selection = UniCox + ElasticNet + RSF strict 3/3"
)

audit.append(
    "Holdout used for Dynamic9 feature selection: NO"
)

audit.append(
    ""
)

audit.append(
    f"Primary external-validation cutoff: "
    f"{EXTERNAL_PRIMARY_CUTOFF}"
)

audit.append(
    ""
)

audit.append(
    "External feature selection allowed: NO"
)

audit.append(
    "External coefficient refitting allowed: NO"
)

audit.append(
    "External hyperparameter tuning allowed: NO"
)

audit.append(
    "External cutoff optimization allowed: NO"
)

audit.append(
    ""
)

audit.append(
    "STEP16 STATUS: COMPLETE"
)

audit.append(
    "EXTERNAL VALIDATION MODEL STATUS: FROZEN"
)


audit_text = "\n".join(
    audit
)


with open(
    AUDIT_DIR /
    "STEP16_FINAL_MODEL_FREEZE_AUDIT.md",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        audit_text
    )


print(
    audit_text
)

In [ ]:
# ============================================================
# OUTPUT MANIFEST
# ============================================================

records = []


for category, folder in [

    (
        "Results",
        RESULTS_DIR
    ),

    (
        "Models",
        MODELS_DIR
    ),

    (
        "Figures",
        FIGURES_DIR
    ),

    (
        "Audit",
        AUDIT_DIR
    )
]:

    for path in sorted(
        folder.rglob("*")
    ):

        if path.is_file():

            records.append({

                "Category":
                    category,

                "File":
                    path.name,

                "Path":
                    str(path),

                "Size_KB":
                    round(
                        path.stat().st_size
                        / 1024,
                        2
                    )
            })


output_manifest = pd.DataFrame(
    records
)


output_manifest.to_csv(
    AUDIT_DIR /
    "04_STEP16_Output_Manifest.csv",
    index=False,
    encoding="utf-8-sig"
)


display(
    output_manifest
)